In [ ]:
# %reset -f
%load_ext autoreload
%autoreload complete --log


In [ ]:
from smtgraphformer import *
from smtgraphformer.smtGraphFormer import SMTConfig
from trainModel import locateModels, locateWorkspace, main, parseConfig

setDisplayOptions()
sr = setReproducibility(17711)


### Setup Ablations

In [ ]:
def slicer(d: SMTConfig) -> dict:
    return {k.replace("use_", ""): v for k, v in vars(d).items() if k.startswith("use_")}


In [ ]:
fp_baseline = "../configs/trainModel.baseline.yaml"
fp_ablation = "../configs/trainModel.bak.yaml"

cfg_baseline = yamlLoader(fp_baseline)
cfg_baseline["model"]["model_dir"] = "../models/ablation"

targets = [
    None,  # retrain baseline for comparison
    "use_graph_embeddings",
    "use_stop_features",
    "use_context_encoding",
    "use_surrogate_tasks",
    "use_mmoe",
]

for t in targets:
    t_config = deepcopy(cfg_baseline)

    print(f"\n--- Ablation: {t} ---")
    # update config for ablation study
    if t is not None:
        t_config["model"][t] = False

    # save updated config to file
    with open(fp_ablation, "w") as f:
        yaml.dump(t_config, f, indent=2, sort_keys=False)

    # train model with updated config
    print(slicer(parseConfig(fp_ablation)[0]))
    !python trainModel.py -c {fp_ablation}

    break  # remove this break to run all ablation studies


In [ ]:
# create a config with all targets ablated for comparison
fp_stripped = "../configs/trainModel.bak.yaml"
stripped = deepcopy(cfg_baseline)
for t in targets[1:]:
    stripped["model"][t] = False

with open(fp_stripped, "w") as f:
    yaml.dump(stripped, f)

print(f"\n--- Ablation: all targets ---")
print(slicer(parseConfig(fp_stripped)[0]))
!python trainModel.py -c {fp_stripped}


In [ ]:
results = summariseFolderResults("../models/")
print(results.head())


### end